# UAS Pembelajaran Mesin — Klasifikasi Kelulusan Mahasiswa
## KNN · Naive Bayes · SVM · GridSearchCV · SMOTE · Feature Selection

**Mata Kuliah:** Pembelajaran Mesin  
**Universitas:** Universitas Dian Nuswantoro  
**Dosen:** Junta Zeniarja, M.Kom  
**Semester:** Genap 2025/2026  

---

### Struktur Notebook
1. Setup & Import
2. SOAL 02 — Audit Dataset & Preprocessing
3. SOAL 03 — Baseline KNN, Naive Bayes, SVM
4. SOAL 04 — Optimasi & Error Analysis
5. Simpan Model Terbaik


## 0. Setup & Import Library

In [ ]:
import os, sys, json, warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import (train_test_split, StratifiedKFold,
                                      GridSearchCV, cross_val_score)
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.feature_selection import SelectKBest, mutual_info_classif
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, balanced_accuracy_score,
                              confusion_matrix, classification_report,
                              ConfusionMatrixDisplay, roc_auc_score)
from imblearn.over_sampling import SMOTE
import joblib

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
print('Libraries loaded successfully')

## SOAL 02 — Audit Dataset, Preprocessing & Pipeline Data

In [ ]:
# Load dataset
df_raw = pd.read_csv('../data/data.csv', sep=';')
df_raw.columns = [c.strip().replace('\t','') for c in df_raw.columns]
print(f'Shape: {df_raw.shape}')
df_raw.head(3)

In [ ]:
# === AUDIT: Basic Info ===
print('=== INFORMASI DASAR ===')
print(f'Jumlah Baris    : {df_raw.shape[0]}')
print(f'Jumlah Kolom    : {df_raw.shape[1]}')
print(f'Duplikat        : {df_raw.duplicated().sum()}')
print(f'Total Missing   : {df_raw.isnull().sum().sum()}')
print()
print('=== DISTRIBUSI TARGET ===')
print(df_raw['Target'].value_counts())
print()
print('Persentase:')
print((df_raw['Target'].value_counts(normalize=True)*100).round(2))

In [ ]:
# Visualisasi distribusi target
fig, axes = plt.subplots(1,2, figsize=(12,4))
vc = df_raw['Target'].value_counts()
colors = ['#28a745', '#dc3545', '#ffc107']
vc.plot(kind='bar', ax=axes[0], color=colors, edgecolor='white')
axes[0].set_title('Distribusi Target (All Classes)', fontweight='bold')
axes[0].set_ylabel('Jumlah'); axes[0].tick_params(axis='x', rotation=0)
for bar in axes[0].patches:
    axes[0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+15,
                 str(int(bar.get_height())), ha='center', fontsize=10)
vc2 = df_raw[df_raw['Target'].isin(['Graduate','Dropout'])]['Target'].value_counts()
vc2.plot(kind='pie', ax=axes[1], autopct='%1.1f%%', colors=['#28a745','#dc3545'],
         startangle=90, wedgeprops={'edgecolor':'white'})
axes[1].set_title('Binary: Graduate vs Dropout', fontweight='bold')
axes[1].set_ylabel('')
plt.tight_layout()
os.makedirs('../reports', exist_ok=True)
plt.savefig('../reports/target_distribution.png', dpi=150)
plt.show(); print('Plot saved.')

In [ ]:
# Outlier audit
numeric_cols = df_raw.select_dtypes(include='number').columns
outlier_report = {}
for col in numeric_cols:
    Q1, Q3 = df_raw[col].quantile(0.25), df_raw[col].quantile(0.75)
    IQR = Q3 - Q1
    n_out = int(((df_raw[col] < Q1-1.5*IQR) | (df_raw[col] > Q3+1.5*IQR)).sum())
    if n_out > 0: outlier_report[col] = n_out
print('=== OUTLIERS IQR — Top 10 ===')
outlier_df = pd.DataFrame.from_dict(outlier_report, orient='index', columns=['n_outlier'])
print(outlier_df.sort_values('n_outlier', ascending=False).head(10))

In [ ]:
# Distribusi fitur utama per kelas
df_binary = df_raw[df_raw['Target'].isin(['Graduate','Dropout'])].copy()
key_features = [
    'Curricular units 1st sem (approved)',
    'Curricular units 2nd sem (approved)',
    'Curricular units 1st sem (grade)',
    'Curricular units 2nd sem (grade)',
    'Age at enrollment', 'Admission grade'
]
fig, axes = plt.subplots(2, 3, figsize=(15,8))
axes = axes.flatten()
for i, feat in enumerate(key_features):
    for label, color in [('Graduate','#28a745'),('Dropout','#dc3545')]:
        axes[i].hist(df_binary[df_binary['Target']==label][feat].dropna(),
                     bins=25, alpha=0.6, color=color, label=label, edgecolor='white')
    axes[i].set_title(feat, fontsize=9, fontweight='bold')
    axes[i].legend(fontsize=8); axes[i].set_ylabel('Frekuensi')
plt.suptitle('Distribusi Fitur Utama berdasarkan Status Kelulusan', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/feature_distributions.png', dpi=150)
plt.show(); print('Plot saved.')

### Preprocessing Pipeline

In [ ]:
# Preprocessing
df = df_binary.copy().drop_duplicates()
df['Target_enc'] = df['Target'].map({'Graduate':1,'Dropout':0})
X = df.drop(columns=['Target','Target_enc'])
y = df['Target_enc']
feature_names = X.columns.tolist()
print(f'Shape setelah filter binary: {df.shape}')
print(f'Distribusi target: {y.value_counts().to_dict()}')

# Stratified split
X_train_raw, X_test_raw, y_train_raw, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_SEED, stratify=y)
print(f'Train: {X_train_raw.shape} | Test: {X_test_raw.shape}')

In [ ]:
# SMOTE
smote = SMOTE(random_state=RANDOM_SEED)
X_train_res, y_train = smote.fit_resample(X_train_raw, y_train_raw)
print(f'Sebelum SMOTE: {dict(zip(*np.unique(y_train_raw, return_counts=True)))}')
print(f'Setelah SMOTE: {dict(zip(*np.unique(y_train, return_counts=True)))}')
print(f'Train shape setelah SMOTE: {X_train_res.shape}')

In [ ]:
# StandardScaler
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train_res)
X_test  = scaler.transform(X_test_raw)
print(f'X_train scaled: {X_train.shape} | X_test scaled: {X_test.shape}')

# Tabel Before/After
before_after = pd.DataFrame({
    'Aspek': ['Rows','Classes','Duplicates','Missing','SMOTE','Scaling'],
    'Sebelum': [df_raw.shape[0],'3','Yes','0','No','No'],
    'Sesudah': [df.shape[0],'2 (binary)','0','0','Yes','Yes (StandardScaler)']
})
print(before_after.to_string(index=False))

## SOAL 03 — Baseline KNN, Naive Bayes, dan SVM

In [ ]:
def evaluate(model, Xt, yt, name):
    yp = model.predict(Xt)
    try: auc = round(roc_auc_score(yt, model.predict_proba(Xt)[:,1]),4)
    except: auc = 'N/A'
    return {
        'model': name,
        'accuracy': round(accuracy_score(yt,yp),4),
        'precision_macro': round(precision_score(yt,yp,average='macro',zero_division=0),4),
        'recall_macro': round(recall_score(yt,yp,average='macro',zero_division=0),4),
        'f1_macro': round(f1_score(yt,yp,average='macro',zero_division=0),4),
        'balanced_accuracy': round(balanced_accuracy_score(yt,yp),4),
        'roc_auc': auc,
        'cm': confusion_matrix(yt,yp),
        'report': classification_report(yt,yp,target_names=['Dropout','Graduate'],zero_division=0)
    }

In [ ]:
# Train baseline models
knn_base = KNeighborsClassifier(n_neighbors=5)
knn_base.fit(X_train, y_train)
knn_res = evaluate(knn_base, X_test, y_test, 'KNN (Baseline)')

nb_base = GaussianNB(var_smoothing=1e-9)
nb_base.fit(X_train, y_train)
nb_res = evaluate(nb_base, X_test, y_test, 'Naive Bayes (Baseline)')

svm_base = SVC(kernel='rbf', C=1.0, gamma='scale', probability=True, random_state=RANDOM_SEED)
svm_base.fit(X_train, y_train)
svm_res = evaluate(svm_base, X_test, y_test, 'SVM (Baseline)')

print('Model     Acc     F1     BalAcc   AUC')
for r in [knn_res, nb_res, svm_res]:
    print(f"{r['model']:25s} {r['accuracy']} {r['f1_macro']} {r['balanced_accuracy']} {r['roc_auc']}")

In [ ]:
# Confusion matrices
fig, axes = plt.subplots(1,3, figsize=(15,4))
for ax, res in zip(axes, [knn_res, nb_res, svm_res]):
    ConfusionMatrixDisplay(res['cm'], display_labels=['Dropout','Graduate']).plot(
        ax=ax, colorbar=False, cmap='Blues')
    ax.set_title(res['model'], fontweight='bold')
plt.suptitle('Confusion Matrix Baseline Models', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/cm_baseline_all.png', dpi=150)
plt.show()

In [ ]:
# Tabel baseline
baseline_df = pd.DataFrame([
    {k:v for k,v in r.items() if k not in ['cm','report']}
    for r in [knn_res, nb_res, svm_res]
])
print(baseline_df.to_string(index=False))

# Cross-validation
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)
print('\n=== 5-Fold CV (macro-F1) ===')
for name, model in [('KNN',knn_base),('Naive Bayes',nb_base),('SVM',svm_base)]:
    scores = cross_val_score(model, X_train, y_train, cv=skf, scoring='f1_macro')
    print(f'{name}: mean={scores.mean():.4f}  std={scores.std():.4f}')

In [ ]:
# Classification reports detail
for r in [knn_res, nb_res, svm_res]:
    print(f"--- {r['model']} ---")
    print(r['report'])
    print()

### Karakteristik Model

| Model | Scaling Sensitive | Asumsi | Kompleksitas | Interpretabilitas |
|-------|:-:|---|---|---|
| KNN | Tinggi | Tidak ada | O(n) inference | Rendah |
| Naive Bayes | Rendah | Fitur independen | O(1) | Tinggi |
| SVM | Tinggi | Margin maksimal | O(n²–n³) training | Rendah-Sedang |

## SOAL 04 — Optimasi Model: GridSearchCV + Feature Selection + Error Analysis

In [ ]:
# GridSearchCV - KNN
param_knn = {'n_neighbors':[3,5,7,9,11,15],'weights':['uniform','distance'],'metric':['euclidean','manhattan']}
gs_knn = GridSearchCV(KNeighborsClassifier(), param_knn, cv=skf, scoring='f1_macro', n_jobs=-1)
gs_knn.fit(X_train, y_train)
knn_opt = gs_knn.best_estimator_
knn_opt_res = evaluate(knn_opt, X_test, y_test, 'KNN (Optimized)')
print(f'KNN Best: {gs_knn.best_params_}')
print(f"F1={knn_opt_res['f1_macro']} BalAcc={knn_opt_res['balanced_accuracy']}")

In [ ]:
# GridSearchCV - Naive Bayes
param_nb = {'var_smoothing':[1e-11,1e-10,1e-9,1e-8,1e-7,1e-6]}
gs_nb = GridSearchCV(GaussianNB(), param_nb, cv=skf, scoring='f1_macro', n_jobs=-1)
gs_nb.fit(X_train, y_train)
nb_opt = gs_nb.best_estimator_
nb_opt_res = evaluate(nb_opt, X_test, y_test, 'Naive Bayes (Optimized)')
print(f'NB Best: {gs_nb.best_params_}')
print(f"F1={nb_opt_res['f1_macro']} BalAcc={nb_opt_res['balanced_accuracy']}")

In [ ]:
# GridSearchCV - SVM
param_svm = {'C':[0.1,1,10],'kernel':['rbf'],'gamma':['scale','auto']}
gs_svm = GridSearchCV(SVC(probability=True,random_state=RANDOM_SEED),
                      param_svm, cv=skf, scoring='f1_macro', n_jobs=-1)
gs_svm.fit(X_train, y_train)
svm_opt = gs_svm.best_estimator_
svm_opt_res = evaluate(svm_opt, X_test, y_test, 'SVM (Optimized)')
print(f'SVM Best: {gs_svm.best_params_}')
print(f"F1={svm_opt_res['f1_macro']} BalAcc={svm_opt_res['balanced_accuracy']}")

In [ ]:
# Feature Selection - Mutual Information top-15
selector = SelectKBest(score_func=mutual_info_classif, k=15)
X_train_sel = selector.fit_transform(X_train, y_train)
X_test_sel  = selector.transform(X_test)
selected_idx = selector.get_support(indices=True)
selected_feats = [feature_names[i] for i in selected_idx]
print('=== TOP-15 SELECTED FEATURES ===')
for i, f in enumerate(selected_feats, 1): print(f'  {i:2d}. {f}')

In [ ]:
# Feature importance plot
mi_df = pd.DataFrame({'feature':feature_names,'score':selector.scores_})
mi_df = mi_df.sort_values('score', ascending=False).head(15)
fig, ax = plt.subplots(figsize=(10,6))
ax.barh(mi_df['feature'][::-1], mi_df['score'][::-1], color='#4C72B0', edgecolor='white')
ax.set_xlabel('Mutual Information Score')
ax.set_title('Top-15 Fitur Paling Informatif', fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/feature_importance.png', dpi=150)
plt.show()

In [ ]:
# SVM + Feature Selection
gs_svm_sel = GridSearchCV(SVC(probability=True,random_state=RANDOM_SEED),
                          param_svm, cv=skf, scoring='f1_macro', n_jobs=-1)
gs_svm_sel.fit(X_train_sel, y_train)
svm_sel = gs_svm_sel.best_estimator_
svm_sel_res = evaluate(svm_sel, X_test_sel, y_test, 'SVM+FeatureSelection')
print(f"F1={svm_sel_res['f1_macro']} BalAcc={svm_sel_res['balanced_accuracy']}")

In [ ]:
# Tabel komparatif
all_results = [knn_res, nb_res, svm_res, knn_opt_res, nb_opt_res, svm_opt_res, svm_sel_res]
comp_df = pd.DataFrame([
    {k:v for k,v in r.items() if k not in ['cm','report']}
    for r in all_results
])
print('=== TABEL KOMPARATIF BASELINE vs OPTIMIZED ===')
print(comp_df.to_string(index=False))
comp_df.to_csv('../reports/all_experiment_results.csv', index=False)
print('Saved all_experiment_results.csv')

In [ ]:
# Grafik perbandingan
metrics = ['accuracy','f1_macro','balanced_accuracy']
fig, axes = plt.subplots(1,3,figsize=(17,5))
palette = sns.color_palette('husl', len(all_results))
for i,m in enumerate(metrics):
    ax = axes[i]
    bars = ax.bar(comp_df['model'], comp_df[m], color=palette, edgecolor='white')
    ax.set_title(m.replace('_',' ').title(), fontweight='bold')
    ax.set_ylim(0,1.15); ax.tick_params(axis='x',rotation=50); ax.set_ylabel('Score')
    for bar,val in zip(bars, comp_df[m]):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.01,
                f'{val:.3f}', ha='center', fontsize=7)
plt.suptitle('Perbandingan Model: Baseline vs Optimized', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/metric_comparison.png', dpi=150)
plt.show()

In [ ]:
# Error Analysis
y_pred_svm = svm_opt.predict(X_test)
err_df = X_test_raw.copy()
err_df['y_true'] = y_test.values
err_df['y_pred'] = y_pred_svm
err_df['true_label'] = err_df['y_true'].map({1:'Graduate',0:'Dropout'})
err_df['pred_label'] = err_df['y_pred'].map({1:'Graduate',0:'Dropout'})
err_df['correct'] = err_df['y_true'] == err_df['y_pred']
errors = err_df[~err_df['correct']]
print(f'Salah prediksi: {len(errors)} / {len(err_df)} ({len(errors)/len(err_df)*100:.1f}%)')
print('\n=== POLA ERROR ===')
print(errors.groupby(['true_label','pred_label']).size().reset_index(name='count'))
print('\n=== FITUR PADA ERROR CASES (statistik) ===')
print(errors[['Curricular units 1st sem (approved)',
               'Curricular units 2nd sem (approved)',
               'Curricular units 1st sem (grade)',
               'Curricular units 2nd sem (grade)']].describe().round(3))

In [ ]:
# Confusion matrices optimized
fig, axes = plt.subplots(1,3, figsize=(15,4))
for ax, res in zip(axes, [knn_opt_res, nb_opt_res, svm_opt_res]):
    ConfusionMatrixDisplay(res['cm'], display_labels=['Dropout','Graduate']).plot(
        ax=ax, colorbar=False, cmap='Greens')
    ax.set_title(res['model'], fontweight='bold')
plt.suptitle('Confusion Matrix Optimized Models', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/cm_optimized_all.png', dpi=150)
plt.show()

### Penentuan Model Terbaik

Berdasarkan eksperimen, **SVM (kernel RBF, C=1)** dipilih sebagai model terbaik karena:
1. F1-Macro tertinggi (0.9021–0.9023) secara konsisten
2. Balanced Accuracy tertinggi (~0.89) — tidak bias pada kelas mayoritas
3. ROC-AUC tinggi (>0.95)
4. Stabil pada cross-validation (5-fold)

Trade-off: SVM lebih lambat dilatih dibanding KNN/NB, tetapi inference cepat setelah training selesai.

In [ ]:
# Simpan model terbaik
best_bundle = {
    'model': svm_opt,
    'scaler': scaler,
    'feature_names': feature_names,
    'model_name': 'SVM',
    'selected_features': selected_feats
}
os.makedirs('../models', exist_ok=True)
joblib.dump(best_bundle, '../models/best_student_graduation_model.joblib')
print('Model bundle saved: best_student_graduation_model.joblib')

# Simpan classification reports
reports = {
    'baseline': {r['model']: r['report'] for r in [knn_res, nb_res, svm_res]},
    'optimized': {r['model']: r['report'] for r in [knn_opt_res, nb_opt_res, svm_opt_res]}
}
with open('../reports/classification_reports.json','w',encoding='utf-8') as f:
    json.dump(reports, f, indent=2, ensure_ascii=False)
print('classification_reports.json saved.')